# Week 3, day 4 (morning) — Worksheet 11 NOTES: designing a star schema

**There is no correct answer to this activity**, so this is not a solution — it
is a second worked example, plus what a good answer looks like at each step and
the mistakes that show up most often.

The example runs a **ride sharing** domain end to end, so you can compare a
finished design against your own. Read it after your group has produced
something, not before.

Nothing here was executed. It is a design, like yours.

STEP 1 — choose an industry

### Question 1

Write **one sentence** naming your industry and the single business process you will model. Not the company — one process.
> **NOTE:** if your sentence contains "and", you are probably modelling two processes. Pick one.

**A good answer is one clause long and names an event.**

> *Ride sharing: a passenger takes a completed trip with a driver.*

That is a repeatable event with a moment in time, a set of participants, and
things worth measuring. Everything else follows from it.

**What goes wrong:**

*"Ride sharing: we want to understand our business."* — not a process, and it
gives you nothing to put in a fact table.

*"Ride sharing: trips, driver payouts and customer support tickets."* — three
processes. Each has a different grain and needs its own fact table. Model one; if
you have time, note that the others would share `dim_date` and `dim_driver`,
which makes it a galaxy schema (worksheet 04).

*"Ride sharing: drivers."* — a driver is a *thing*, not an event. Things become
dimensions. If your sentence names a noun rather than something happening, you
have found a dimension and not yet found your process.

**The test:** can you finish the sentence *"every time this happens, we record
..."*? If not, keep looking.

STEP 2 — identify the data

### Question 2

Brainstorm what this process generates. Answer three questions: what event happens over and over? What gets measured each time? What descriptive details surround each event? List everything; sort it in step 3.

**A good answer is a messy list, produced fast.** Do not sort it yet — sorting
is step 3, and sorting while brainstorming stops the brainstorming.

For the ride sharing example:

**The repeating event:** a trip is completed.

**What gets measured each time:** fare, distance, duration, surge multiplier,
driver payout, platform commission, tip, wait time before pickup, passenger
rating, driver rating.

**Descriptive details around each event:** the passenger (name, signup date,
city, membership tier), the driver (name, rating, vehicle, city, joined date),
the vehicle (make, model, year, category — economy/XL/premium), pickup and
dropoff location (city, neighbourhood, airport flag), the date and time (day of
week, hour, holiday flag, peak/off-peak), payment method, promotion applied,
trip status.

**Two things to notice in that list**, because they matter in step 3:

`surge multiplier` and `passenger rating` are numbers, and neither is a
measure you can add up. Worksheet 03 — a multiplier and a rating are
non-additive. They may still belong somewhere; they do not belong in a `SUM`.

`peak/off-peak` sounds like a fact about the trip and is really a fact about the
**time**. It belongs in `dim_date` (or a `dim_time`), where it is computed once,
rather than being recorded per trip where two rows can disagree.

**Twenty items in five minutes is the right pace.** You will discard half.

STEP 3 — separate facts from dimensions

### Question 3

**State the grain.** What does one row of your fact table represent? One sentence. Decide this before anything else.
> **NOTE:** worksheet 02 — a grain is a uniqueness claim. Say what combination of things identifies exactly one row.

**The grain for the ride sharing example:**

> **One row represents one completed trip.**

Test it as a uniqueness claim, which is what worksheet 02 established a grain
actually is: *is `trip_id` unique?* If yes, that is your grain and your key.

**Alternative grains, and what each costs:**

| grain | rows | what you lose |
|---|---|---|
| one row per trip | most | nothing — the finest the source supports |
| one row per driver per day | fewer | individual trips, pickup locations, exact times |
| one row per city per hour | fewest | everything about who was involved |

Worksheet 02 question 5's rule: **build at the finest grain the source supports,
and aggregate from it.** A trip-grain table can produce the daily summary; the
daily summary can never produce the trips.

**The trap to check for.** If your process has anything like a "line item" — a
trip with multiple stops, an order with multiple products, a policy with multiple
claimants — then *one row per order* and *one row per line* are different grains
and the difference is not cosmetic. Worksheet 01 question 6 measured what happens
when a measure sits at the wrong one: revenue overstated by 2.16x.

**Write the sentence down.** Slide 41 asks for it in the specification, and
worksheet 02 question 4 showed why: the grain is not recoverable from the code,
and getting it wrong is invisible in the output.

### Question 4

**List the measures** — the numeric values recorded at that grain. For each one, mark it `additive`, `semi-additive` or `non-additive`.
> **NOTE:** worksheet 03 — a snapshot of a level (balance, stock, headcount) is semi-additive. A ratio or percentage is non-additive and should not be stored at all.

**Measures for `FACT_TRIP`, at one row per completed trip:**

| Measure | Type | Note |
|---|---|---|
| `trip_count` | additive | constant 1 — worksheet 07 question 1 |
| `fare_amount` | additive | what the passenger was charged |
| `discount_amount` | additive | promotion applied |
| `net_fare_amount` | additive | `fare_amount - discount_amount` |
| `driver_payout` | additive | |
| `platform_commission` | additive | |
| `tip_amount` | additive | |
| `distance_km` | additive | |
| `duration_minutes` | additive | |
| `wait_time_minutes` | additive | |

**Deliberately not stored:**

`surge_multiplier` — **non-additive**. Summing it is meaningless and averaging it
is misleading, for the reason worksheet 03 question 6 established: an average of
per-trip multipliers is not the same as the ratio of surged fare to base fare, and
the two disagree. Store `base_fare_amount` and `fare_amount` as separate additive
measures and derive the multiplier in the query.

`passenger_rating` and `driver_rating` — **non-additive**, and worse, they are
usually snapshots of a running average rather than a per-trip measurement. A
per-trip *star rating given* is a legitimate additive-ish measure with a count
beside it; a driver's *current* 4.87 average is an attribute of the driver, and it
belongs on `dim_driver` — where worksheet 08's Type 1 caveat applies, since
overwriting it destroys history.

**Where a semi-additive measure would appear:** `drivers_online` at the end of each
hour, in a separate hourly-snapshot fact table. Additive across cities, meaningless
summed across hours — worksheet 03 questions 4 and 5.

**The rule from worksheet 03 question 6:** store the numerator and the
denominator; derive the ratio. Every ratio in your list should decompose into two
additive measures.

### Question 5

**List your dimensions** — three to five tables. Name each one and say in a few words what it describes.

**Five dimensions for `FACT_TRIP`:**

| Dimension | Describes |
|---|---|
| `DIM_DATE` | when the trip happened — day, week, month, quarter, year, holiday flag |
| `DIM_PASSENGER` | who took the trip |
| `DIM_DRIVER` | who drove it |
| `DIM_VEHICLE` | what they drove |
| `DIM_LOCATION` | where it started (and, via a second key, where it ended) |

Five is the top of the activity's 3-5 range and it is a defensible five: each one
is a thing people will filter and group by, and none of them is a measure in
disguise.

**Three design points worth arguing about in your group:**

**`DIM_DATE` always earns its place.** Worksheet 08 question 5 — it turns every
`EXTRACT(QUARTER FROM ...)` into a join to a column, so every query agrees on what
a quarter is. Generate every calendar day, not only the days with trips, or you
cannot find the days with none.

**`DIM_LOCATION` appears twice on the fact table.** `pickup_location_key` and
`dropoff_location_key` both point at the same dimension. That is a **role-playing
dimension**, it is completely standard, and it is the answer to "do I need two
location tables?" — no. The same table, joined twice, aliased differently in the
query. `DIM_DATE` often plays several roles too: booked date, trip date, paid
date.

**`DIM_VEHICLE` might not deserve to exist.** If every driver has exactly one
vehicle, its attributes could sit on `DIM_DRIVER` and you would have four
dimensions and one fewer join. It deserves its own table when drivers switch
vehicles, when vehicles are fleet-owned and shared, or when you want to analyse by
vehicle independently of driver. **Decide on the business reality, not on
tidiness.**

### Question 6

For each dimension, list a few **example attributes**. Include at least one hierarchy (something like `region -> store` or `category -> product`).

**Attributes, with the hierarchies marked:**

```
DIM_DATE       date_key (PK), full_date, day_of_week, day, week, month,
               quarter, year, is_weekend, is_holiday
               hierarchy: year -> quarter -> month -> date

DIM_PASSENGER  passenger_key (PK), source_passenger_id, passenger_name,
               signup_date, membership_tier, home_city

DIM_DRIVER     driver_key (PK), source_driver_id, driver_name, joined_date,
               driver_status, home_city, current_rating

DIM_VEHICLE    vehicle_key (PK), source_vehicle_id, make, model, year,
               vehicle_category, seat_count
               hierarchy: category -> make -> model

DIM_LOCATION   location_key (PK), source_location_id, neighbourhood, city,
               state, country, is_airport
               hierarchy: country -> state -> city -> neighbourhood
```

**Three things this list does deliberately.**

**Every dimension has both a surrogate key and a source key.** Worksheet 08
question 9 — `passenger_key` is the warehouse's own, `source_passenger_id` is what
the operational system calls it. That is what makes a row traceable back to the
source, survivable across a source renumbering, and capable of having history at
all.

**The hierarchies are flattened, not split.** `DIM_LOCATION` holds
`neighbourhood`, `city`, `state` and `country` in one row, with `country`
repeated across thousands of rows. That is a **star**, and worksheet 04 measured
the trade: 5x the storage for a category name, against 1 join instead of 3. At
dimension scale, the star wins.

**`current_rating` on `DIM_DRIVER` is a liability worth naming.** It changes
constantly, and a Type 1 overwrite means last quarter's report shows this
quarter's ratings. If ratings matter analytically, either accept that (and say so)
or make `DIM_DRIVER` a Type 2 dimension with validity dates — which is only
possible because the surrogate key is separate from the source key.

STEP 4 — sketch the star

### Question 7

**Draw the ER diagram.** Fact table in the centre, each dimension connected to it by a line. Show table names and key columns. Paste a screenshot or image link here.
> **NOTE:** aim for a clean star — every dimension connects directly to the one fact table. If a dimension connects to another dimension, you have snowflaked; that may be fine, but say why.

```
                              DIM_DATE
                                  |
        DIM_PASSENGER ---- FACT_TRIP ---- DIM_DRIVER
                              /   \
                             /     \
                   DIM_VEHICLE     DIM_LOCATION
                                    (x2: pickup, dropoff)


FACT_TRIP
  trip_id                  (PK, degenerate)
  date_key                 (FK) -> DIM_DATE
  passenger_key            (FK) -> DIM_PASSENGER
  driver_key               (FK) -> DIM_DRIVER
  vehicle_key              (FK) -> DIM_VEHICLE
  pickup_location_key      (FK) -> DIM_LOCATION
  dropoff_location_key     (FK) -> DIM_LOCATION
  ------------------------------------------
  trip_count               1 per row
  fare_amount
  discount_amount
  net_fare_amount
  driver_payout
  platform_commission
  tip_amount
  distance_km
  duration_minutes
  wait_time_minutes
```

**What makes this a clean star:** every dimension connects directly to
`FACT_TRIP`. Nothing connects to anything else. Any attribute in the model is one
join from any measure.

**Four things a marker looks for.**

**`trip_id` is in the fact table with no dimension behind it.** That is a
*degenerate dimension* — worksheet 03 question 1. It identifies the row, it is not
a measure, and there are no attributes to factor out. Keep it: it is what makes a
fact row traceable to the source record when a number looks wrong.

**Two keys to one dimension.** `pickup_location_key` and `dropoff_location_key`
both point at `DIM_LOCATION`. Drawing that as one line with two labels is correct
and shows you understand role-playing dimensions.

**No text in the fact table.** No `driver_name`, no `city`, no `vehicle_category`.
Worksheet 03 question 8 measured the cost of getting this wrong: 93x the storage
and 114 rows to update instead of 1.

**The measures are separated from the keys visually.** A fact table read top to
bottom should say *what happened*, then *how much*.

Your deliverable is a real ER diagram, not ASCII. draw.io, Lucidchart,
dbdiagram.io or a photo of a whiteboard are all fine — the diagram is what you
present from.

VALIDATE — worksheet 10's step, in miniature

### Question 8

Write **three business questions** your model should answer, then say for each which measure you would sum and which dimensions you would group by. If one of your questions cannot be answered, fix the model or write down why not.

**Three questions, and how the model answers them:**

| Business question | Measure | Group by |
|---|---|---|
| How many trips per day, and is it growing? | `SUM(trip_count)` | `DIM_DATE.full_date` |
| Which neighbourhoods generate the most revenue? | `SUM(net_fare_amount)` | `DIM_LOCATION.neighbourhood` via `pickup_location_key` |
| Do premium vehicles get better tips? | `SUM(tip_amount) / SUM(net_fare_amount)` | `DIM_VEHICLE.vehicle_category` |

All three work, and the third one is worth reading closely — it is
`SUM(numerator) / SUM(denominator)`, **not** `AVG(tip_rate)`. Worksheet 03
question 6: those are different numbers, and only one of them is about money. It
is computable precisely because both measures are stored additive and the ratio is
derived in the query.

**Now the question that fails**, and every group should find one:

> *What is our average driver utilisation — the share of time drivers are
> online but not on a trip?*

`FACT_TRIP` has one row per **trip**. It knows nothing about the time between
trips, or about a driver who was online all evening and got no bookings. **The
events this model records are trips; idle time is the absence of an event, and you
cannot count absences in a table of occurrences.**

Answering it needs a second fact table at a different grain — one row per driver
per hour, with `minutes_online` and `minutes_on_trip` — sharing `DIM_DATE` and
`DIM_DRIVER`. That is a galaxy schema (worksheet 04 question 6), and
`minutes_online` in an hourly snapshot is the semi-additive measure your first
model did not have.

**Finding this at design time costs five minutes. Finding it after the dashboard
is built costs a rebuild.** That is the whole argument for slide 30's validation
step.

### Question 9

**The argument you had.** Name one attribute your group was unsure about — measure or dimension, or which dimension it belongs to — and record which way you decided and why.
> **NOTE:** the activity brief asks for this specifically. It is usually the most interesting part of the design.

The activity brief asks for this specifically, and it is where the real learning
is. Three arguments that come up almost every time:

**`surge_multiplier` — measure or attribute?**

It is a number recorded per trip, so it looks like a measure. It is non-additive,
so it cannot be summed, and averaging it gives a number that disagrees with the
ratio of surged to base revenue. **Resolution:** not a measure. Store
`base_fare_amount` and `fare_amount` — both additive — and derive the multiplier
in the query. If analysts need to *filter* by surge, add a banded attribute
(`surge_band`: none / 1-1.5x / 1.5-2x / 2x+) to a small dimension, where a filter
is what it is for.

**`is_airport_trip` — fact or dimension?**

It is a property of the trip, so it feels like a fact-table flag. It is also
derivable from `DIM_LOCATION.is_airport` on either end. **Resolution:** put
`is_airport` on `DIM_LOCATION` and let the join do it, because that way there is
one definition of "airport" rather than a flag on 50 million fact rows that can
disagree with the location table. The exception is if the *rule* is complex and
changes — then a fact-table flag records what was true when the trip happened,
which is a different and sometimes better answer.

**`membership_tier` — passenger dimension, or fact table?**

A passenger's tier changes over time. Put it on `DIM_PASSENGER` and a Type 1
overwrite means every historical trip retroactively shows their *current* tier —
so last year's "revenue by tier" report changes every time someone upgrades.
**Resolution:** either accept it and document it, or store the tier as it was at
trip time on the fact row, or make `DIM_PASSENGER` a Type 2 dimension.

**What all three have in common:** the answer depends on a business question
nobody in the room can settle, and the deliverable is *the decision plus its
reason*, not the decision alone. Worksheet 06's rule register is the same idea —
the reason is the part that survives.

SUMMARY

### Question 10

Write a short summary of the finished model — three or four sentences. Then name **one question your model cannot answer**, and what you would add to answer it.
> **NOTE:** worksheet 10 question 10. Every model has an edge; a group that cannot find theirs has not looked.

**A summary that would earn full marks:**

> `FACT_TRIP` records one row per completed trip in a ride sharing marketplace,
> at a grain of one trip. It carries ten additive measures — trip count, fare,
> discount, net fare, driver payout, commission, tip, distance, duration and wait
> time — and no ratios: `surge_multiplier` and the rating fields were deliberately
> excluded as non-additive, with their components stored instead so any rate can
> be derived at query time.
>
> Five dimensions attach directly: `DIM_DATE`, `DIM_PASSENGER`, `DIM_DRIVER`,
> `DIM_VEHICLE`, and `DIM_LOCATION`, which plays two roles as pickup and dropoff.
> `trip_id` stays in the fact table as a degenerate dimension for traceability
> back to the source. The model answers trip volume, revenue by geography, and
> tipping by vehicle category with a single join each.
>
> Its main open risk is `DIM_PASSENGER.membership_tier`, which is a Type 1
> overwrite: historical trips will show a passenger's current tier rather than
> their tier at the time. We accepted that for now and recorded it.

**The question it cannot answer**, and what to add:

> **Driver utilisation** — the share of online time not spent on a trip. A trip
> table cannot measure the gaps between trips.
>
> Add `FACT_DRIVER_HOURLY` at one row per driver per hour, with `minutes_online`,
> `minutes_on_trip` and `trips_completed`, sharing `DIM_DATE` and `DIM_DRIVER`.
> That makes the model a galaxy schema. `minutes_online` is semi-additive —
> additive across drivers within an hour, not across hours for one driver — and
> the two fact tables must never be joined directly; aggregate each to a common
> grain first.

---

## What to look for in the share-out

When another group presents, these are the five questions worth asking:

1. **"What does one row mean?"** If the answer takes more than a sentence, or
   contains "or", the grain is not settled.
2. **"Is anything in the fact table text?"** Names and labels belong in
   dimensions.
3. **"Which of your measures can I not sum?"** Every model has at least one, and
   a group that says "all of them are fine" has a ratio hiding in the list.
4. **"Show me a question this cannot answer."** Every model has an edge.
5. **"What did you argue about?"** The most interesting answer of the five.

## The day, in one page

| | |
|---|---|
| **Grain** | a uniqueness claim you test, not a sentence you write |
| **Facts** | measures and keys only — no text |
| **Dimensions** | attributes and hierarchies, flattened, with surrogate + source keys |
| **Additive** | is a property of a measure **and a grain**, not of a column |
| **Non-additive** | store the numerator and denominator; derive the ratio |
| **Star by default** | snowflake only for large or genuinely shared hierarchies |
| **Galaxy** | two facts, conformed dimensions — and never join them directly |
| **Unknown members** | one row per dimension, so nothing is silently dropped |
| **Business rules** | decisions, in a register that computes its own counts |
| **Validate** | reconcile outside the table, then ask the six questions — and a seventh |